# Tutorial 21: YOLO26 Object Detection, Pose, and Segmentation

This tutorial explains a Qt-based C++ application that runs three YOLO26-S models on the same camera or video stream with a DEEPX NPU.

![YOLO26 three-task demo](assets/yolo26-od-pos-seg.png)

## Learning goals

By the end of this tutorial, you will be able to:

- understand the self-contained C++ project structure;
- identify the detection, pose, and segmentation pipelines;
- understand how one input frame is distributed to three workers;
- build the Qt application in Release mode; and
- run the demo with a camera or video file.

## 1. Locate the tutorial files

In [1]:
from pathlib import Path
import shutil
import subprocess

candidates = [
    Path.cwd(),
    Path.cwd() / "notebooks" / "T21-demo-yolo26-od-pos-seg",
]

TUTORIAL_ROOT = next(
    (path.resolve() for path in candidates if (path / "app" / "CMakeLists.txt").is_file()),
    None,
)
if TUTORIAL_ROOT is None:
    raise FileNotFoundError("Could not find the T21-demo-yolo26-od-pos-seg directory.")

APP_ROOT = TUTORIAL_ROOT / "app"
ASSET_ROOT = TUTORIAL_ROOT / "assets"

print(f"Tutorial: {TUTORIAL_ROOT}")
print(f"App:      {APP_ROOT}")
print(f"Assets:   {ASSET_ROOT}")

Tutorial: /home/donggyun/git/dx-tutorials/notebooks/T21-demo-yolo26-od-pos-seg
App:      /home/donggyun/git/dx-tutorials/notebooks/T21-demo-yolo26-od-pos-seg/app
Assets:   /home/donggyun/git/dx-tutorials/notebooks/T21-demo-yolo26-od-pos-seg/assets


### Project layout

```text
T21-demo-yolo26-od-pos-seg/
├── get_resources.sh
├── assets/
│   └── videos/
├── app/
│   ├── build.sh
│   ├── run_camera.sh
│   ├── run_video.sh
│   ├── CMakeLists.txt
│   ├── yolo26s_3.cpp
│   ├── common/
│   │   ├── base/
│   │   ├── processors/
│   │   └── utility/
│   ├── factory/
│   └── extern/
└── yolo26_od_pose_seg.ipynb
```


In [2]:
required_files = [
    TUTORIAL_ROOT / "get_resources.sh",
    APP_ROOT / "build.sh",
    APP_ROOT / "run_camera.sh",
    APP_ROOT / "run_video.sh",
    APP_ROOT / "CMakeLists.txt",
    APP_ROOT / "yolo26s_3.cpp",
    APP_ROOT / "factory" / "yolo26s_factory.hpp",
    APP_ROOT / "factory" / "yolo26s_pose_factory.hpp",
    APP_ROOT / "factory" / "yolo26s_seg_factory.hpp",
    APP_ROOT / "utility" / "common_util.cpp",
]

for path in required_files:
    status = "OK" if path.is_file() else "MISSING"
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_ROOT)}")

[OK     ] get_resources.sh
[OK     ] app/build.sh
[OK     ] app/run_camera.sh
[OK     ] app/run_video.sh
[OK     ] app/CMakeLists.txt
[OK     ] app/yolo26s_3.cpp
[OK     ] app/factory/yolo26s_factory.hpp
[OK     ] app/factory/yolo26s_pose_factory.hpp
[OK     ] app/factory/yolo26s_seg_factory.hpp
[MISSING] app/utility/common_util.cpp


## 2. Check the environment

Install the Debian packages listed in `README.md`. The DEEPX device driver and DXRT SDK must also be installed.

In [3]:
commands = ["g++", "cmake", "make", "qmake", "v4l2-ctl", "dxrt-cli"]
for command in commands:
    location = shutil.which(command)
    print(f"[{'OK' if location else 'MISSING':7}] {command}")

device_nodes = sorted(Path("/dev").glob("dxrt*"))
print(f"\nDEEPX device nodes: {device_nodes if device_nodes else 'not found'}")

[OK     ] g++
[OK     ] cmake
[OK     ] make
[OK     ] qmake
[OK     ] v4l2-ctl
[OK     ] dxrt-cli

DEEPX device nodes: [PosixPath('/dev/dxrt0')]


## 3. Download and check the resources

`get_resources.sh` downloads the resource archive, extracts its models, image, and sample videos into `assets/`, and removes the downloaded archive after successful extraction.

The application expects these files by default:

```text
assets/
├── models/
│   ├── yolo26s.dxnn
│   ├── yolo26s-pose.dxnn
│   └── yolo26s-seg.dxnn
├── yolo26.png
└── videos/
    └── <input-video>
```

In [4]:
default_assets = [
    ASSET_ROOT / "models" / "yolo26s.dxnn",
    ASSET_ROOT / "models" / "yolo26s-pose.dxnn",
    ASSET_ROOT / "models" / "yolo26s-seg.dxnn",
    ASSET_ROOT / "yolo26.png",
]

for path in default_assets:
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_ROOT)}")

video_files = sorted((ASSET_ROOT / "videos").glob("*"))
video_files = [path for path in video_files if path.is_file() and path.name != '.gitkeep']
print(f"\nAvailable video files: {len(video_files)}")
for path in video_files[:10]:
    print(f"  - {path.name}")

resources_missing = any(not path.is_file() for path in default_assets) or not video_files

[OK     ] assets/models/yolo26s.dxnn
[OK     ] assets/models/yolo26s-pose.dxnn
[OK     ] assets/models/yolo26s-seg.dxnn
[OK     ] assets/yolo26.png

Available video files: 1
  - dance-960-540.mp4


Run the next cell only when resources are missing. Existing files with the same names may be replaced during extraction.

In [5]:
if resources_missing:
    subprocess.run(
        [str(TUTORIAL_ROOT / "get_resources.sh")],
        cwd=TUTORIAL_ROOT,
        check=True,
    )
else:
    print("All required resources are already available.")

video_files = sorted((ASSET_ROOT / "videos").glob("*"))
video_files = [path for path in video_files if path.is_file() and path.name != '.gitkeep']
for path in default_assets:
    print(f"[{'OK' if path.is_file() else 'MISSING':7}] {path.relative_to(TUTORIAL_ROOT)}")
print(f"Available video files: {len(video_files)}")

All required resources are already available.
[OK     ] assets/models/yolo26s.dxnn
[OK     ] assets/models/yolo26s-pose.dxnn
[OK     ] assets/models/yolo26s-seg.dxnn
[OK     ] assets/yolo26.png
Available video files: 1


## 4. Application architecture

```text
Camera or video
       |
       v
CaptureThread
       |
       +--> Detection worker    --> Object Detection panel
       +--> Pose worker         --> Pose Estimation panel
       +--> Segmentation worker --> Instance Segmentation panel
                                      + static Demo Image panel
```

The capture thread publishes each BGR frame to three queues. Each `ResultWorker` owns a DXRT inference engine, a task-specific factory, a preprocessor, and a postprocessor. The application renders each task result and displays the three results plus one static image in a full-screen 2 x 2 Qt grid.

## 5. Read the C++ code

The following helper displays selected sections from the current source files.

In [6]:
from IPython.display import Code, display

def show_source(relative_path, marker, line_count=80):
    path = APP_ROOT / relative_path
    lines = path.read_text(encoding="utf-8").splitlines()
    try:
        start = next(index for index, line in enumerate(lines) if marker in line)
    except StopIteration as error:
        raise ValueError(f"Marker not found in {relative_path}: {marker}") from error

    end = min(start + line_count, len(lines))
    print(f"{relative_path}:{start + 1}-{end}")
    display(Code("\n".join(lines[start:end]), language="cpp"))

### 5.1. Build configuration

CMake builds one C++17 executable and links Qt5 Widgets, OpenCV, and DXRT. `PROJECT_ROOT_DIR` points to the tutorial directory so the default model and image paths resolve under `assets/`.

In [7]:
show_source("CMakeLists.txt", "find_package(OpenCV", line_count=55)

CMakeLists.txt:7-40


find_package(OpenCV REQUIRED)
find_package(Qt5 REQUIRED COMPONENTS Widgets)

if(CROSS_COMPILE OR MSVC)
    find_library(DXRT_LIB dxrt HINTS ${DXRT_INSTALLED_DIR}/lib REQUIRED)
    include_directories(${DXRT_INSTALLED_DIR}/include)
    link_directories(${DXRT_INSTALLED_DIR}/lib)
else()
    find_package(dxrt REQUIRED HINTS ${DXRT_INSTALLED_DIR})
    set(DXRT_LIB dxrt)
endif()

if(NOT DEFINED PROJECT_ROOT_OVERRIDE)
    get_filename_component(PROJECT_ROOT_OVERRIDE "${CMAKE_CURRENT_SOURCE_DIR}/.." ABSOLUTE)
endif()

add_executable(yolo26s_3
    yolo26s_3.cpp
)

target_compile_definitions(yolo26s_3 PRIVATE PROJECT_ROOT_DIR="${PROJECT_ROOT_OVERRIDE}")

target_include_directories(yolo26s_3 PRIVATE
    ${OpenCV_INCLUDE_DIRS}
    "${CMAKE_CURRENT_SOURCE_DIR}"
)

target_compile_options(yolo26s_3 PRIVATE -Wall -Wextra -O3)

target_link_libraries(yolo26s_3
    Qt5::Widgets
    ${OpenCV_LIBS}
    ${DXRT_LIB}
)

### 5.2. Command-line options and default resources

`AppArgs` defines the three model paths, demo image, camera settings, optional video path, and debugging options. Omitting `--video` selects camera mode. Use `-c` or `--camera` to choose a V4L2 device and `--width`, `--height`, and `--fps` to request capture settings. If these options are omitted, the defaults are `/dev/video0`, 1280 x 720, and 30 FPS.

In [8]:
show_source("yolo26s_3.cpp", "struct AppArgs", line_count=48)
show_source("yolo26s_3.cpp", "AppArgs parseArgs", line_count=70)

yolo26s_3.cpp:124-171


struct AppArgs {
    std::string model = projectPath("assets/models/yolo26s.dxnn");
    std::string model_pose = projectPath("assets/models/yolo26s-pose.dxnn");
    std::string model_seg = projectPath("assets/models/yolo26s-seg.dxnn");
    std::string demo_image = projectPath("assets/yolo26.png");
    std::string video;
    bool no_loop_video = false;
    std::string device = "/dev/video0";
    int width = 1280;
    int height = 720;
    int fps = 30;
    bool debug_seg_timing = false;
    int debug_timing_interval_ms = 1000;
    bool debug_seg_bbox_only = false;
    int seg_render_width = 640;
    int seg_render_height = 360;
    bool save_video = false;
    std::string output_video;
    bool show_exit_button = false;
};

void printUsage(const char* argv0, const AppArgs& defaults) {
    std::cout
        << "Usage: " << argv0 << " [OPTIONS]\n"
        << "      --model <PATH>          YOLO26 detection .dxnn (default: "
        << defaults.model << ")\n"
        << "      --model-pose <PATH>     YOLO26 pose .dxnn (default: "
        << defaults.model_pose << ")\n"
        << "      --model-seg <PATH>      YOLO26 segmentation .dxnn (default: "
        << defaults.model_seg << ")\n"
        << "      --demo-image <PATH>     Bottom-right panel image (default: "
        << defaults.demo_image << ")\n"
        << "      --video <PATH>          Video file input path (default: use camera)\n"
        << "      --no-loop-video         Stop at EOF instead of looping video\n"
        << "  -c, --camera <PATH>         V4L2 camera device (default: "
        << defaults.device << ")\n"
        << "      --device <PATH>         Deprecated alias for --camera\n"
        << "      --width <N>             Requested camera width (default: "
        << defaults.width << ")\n"
        << "      --height <N>            Requested camera height (default: "
        << defaults.height << ")\n"
        << "      --fps <N>               Requested camera FPS (default: "
        << defaults.fps << ")\n"
        << "      --debug-seg-timing      Print segmentation worker timing by stage\n"
        << "      --debug-timing-interval-ms <N>\n"
        << "                              Timing report interval in ms (default: "
        << defaults.debug_timing_interval_ms << ")\n"
        << "      --debug-seg-bbox-only   Accepted for compatibility\n"

yolo26s_3.cpp:182-251


AppArgs parseArgs(int argc, char* argv[]) {
    AppArgs args;
    std::string legacy_device;
    cxxopts::Options options(
        "yolo26s_3",
        "Qt5 2x2 YOLO26-S demo: object detection, pose, instance segmentation, demo image.");

    options.add_options()
        ("model", "YOLOv26 detection .dxnn",
         cxxopts::value<std::string>(args.model)->default_value(args.model))
        ("model-pose", "YOLOv26 pose .dxnn",
         cxxopts::value<std::string>(args.model_pose)->default_value(args.model_pose))
        ("model-seg", "YOLOv26 segmentation .dxnn",
         cxxopts::value<std::string>(args.model_seg)->default_value(args.model_seg))
        ("demo-image", "Image shown in the bottom-right panel.",
         cxxopts::value<std::string>(args.demo_image)->default_value(args.demo_image))
        ("video", "Video file path input. If omitted, USB webcam is used.",
         cxxopts::value<std::string>(args.video)->default_value(""))
        ("no-loop-video", "With --video, stop at EOF instead of looping.",
         cxxopts::value<bool>(args.no_loop_video)->default_value("false"))
        ("c,camera", "V4L2 camera device, e.g. /dev/video0.",
         cxxopts::value<std::string>(args.device)->default_value(args.device))
        ("device", "Deprecated alias for --camera.",
         cxxopts::value<std::string>(legacy_device))
        ("width", "Requested camera width.",
         cxxopts::value<int>(args.width)->default_value(std::to_string(args.width)))
        ("height", "Requested camera height.",
         cxxopts::value<int>(args.height)->default_value(std::to_string(args.height)))
        ("fps", "Requested camera FPS.",
         cxxopts::value<int>(args.fps)->default_value(std::to_string(args.fps)))
        ("debug-seg-timing", "Print segmentation worker timing by stage.",
         cxxopts::value<bool>(args.debug_seg_timing)->default_value("false"))
        ("debug-timing-interval-ms", "Timing report interval in milliseconds.",
         cxxopts::value<int>(args.debug_timing_interval_ms)->default_value(
             std::to_string(args.debug_timing_interval_ms)))
        ("debug-seg-bbox-only", "Accepted for compatibility; segmentation renders mask colors only.",
         cxxopts::value<bool>(args.debug_seg_bbox_only)->default_value("false"))
        ("seg-render-width", "Segmentation panel render width. Capped at 960.",
         cxxopts::value<int>(args.seg_render_width)->default_value(
             std::to_string(args.seg_render_width)))
        ("seg-render-height", "Segmentation panel render height. Capped at 540.",
         cxxopts::value<int>(args.seg_render_height)->default_value(
             std::to_string(args.seg_render_height)))
        ("s,save-video", "Save output video file. If used without --output-video, saves to 'output.mp4'.",
         cxxopts::value<bool>(args.save_video)->default_value("false"))
        ("output-video", "Output video file path (enables saving when specified).",
         cxxopts::value<std::string>(args.output_video)->default_value(""))
        ("exit-btn", "Show an exit button in the Pose Estimation panel header.",
         cxxopts::value<bool>(args.show_exit_button)->default_value("false"))
        ("h,help", "Print usage");

    auto result = options.parse(argc, argv);
    if (result.count("help")) {
        printUsage(argv[0], args);
        std::exit(0);
    }

    if (result.count("device") > 0 && result.count("camera") == 0) {
        args.device = legacy_device;
    }
    
    // Handle video saving options
    if (!args.output_video.empty()) {
        args.save_video = true;
    } else if (args.save_video && args.output_video.empty()) {
        args.output_video = "output.mp4";
    }
    
    return args;
}

### 5.3. Task factories

Each factory creates the correct preprocessing, post-processing, and visualization components for one task.

In [9]:
show_source("factory/yolo26s_factory.hpp", "class Yolo26sFactory", line_count=42)
show_source("factory/yolo26s_pose_factory.hpp", "class Yolo26s_poseFactory", line_count=42)
show_source("factory/yolo26s_seg_factory.hpp", "class Yolo26s_segFactory", line_count=42)

factory/yolo26s_factory.hpp:15-42


class Yolo26sFactory {
public:
    Yolo26sFactory(float score_threshold = 0.3f,
                   float nms_threshold = 0.45f)
        : score_threshold_(score_threshold),
          nms_threshold_(nms_threshold) {}

    PreprocessorPtr createPreprocessor(int input_width, int input_height) {
        return std::make_unique<DetectionPreprocessor>(input_width, input_height);
    }

    PostprocessorPtr<DetectionResult> createPostprocessor(
        int input_width, int input_height, bool is_ort_configured = false) {
        return std::make_unique<YOLOv26Postprocessor>(
            input_width, input_height,
            score_threshold_, nms_threshold_,
            is_ort_configured
        );
    }

private:
    float score_threshold_;
    float nms_threshold_;
};

}  // namespace dxapp

#endif  // YOLO26S_FACTORY_HPP

factory/yolo26s_pose_factory.hpp:17-44


class Yolo26s_poseFactory {
public:
    Yolo26s_poseFactory(float score_threshold = 0.3f,
                      float nms_threshold = 0.45f)
        : score_threshold_(score_threshold),
          nms_threshold_(nms_threshold) {}

    PreprocessorPtr createPreprocessor(int input_width, int input_height) {
        return std::make_unique<DetectionPreprocessor>(input_width, input_height);
    }

    PostprocessorPtr<PoseResult> createPostprocessor(
        int input_width, int input_height, bool is_ort_configured = false) {
        (void)is_ort_configured;
        return std::make_unique<YOLO26PosePostprocessor>(
            input_width, input_height,
            score_threshold_, nms_threshold_
        );
    }

private:
    float score_threshold_;
    float nms_threshold_;
};

}  // namespace dxapp

#endif  // YOLO26S_POSE_FACTORY_HPP

factory/yolo26s_seg_factory.hpp:15-42


class Yolo26s_segFactory {
public:
    Yolo26s_segFactory(float score_threshold = 0.3f,
                      float nms_threshold = 0.45f)
        : score_threshold_(score_threshold),
          nms_threshold_(nms_threshold) {}

    PreprocessorPtr createPreprocessor(int input_width, int input_height) {
        return std::make_unique<DetectionPreprocessor>(input_width, input_height);
    }

    PostprocessorPtr<InstanceSegmentationResult> createPostprocessor(
        int input_width, int input_height, bool is_ort_configured = false) {
        return std::make_unique<YOLO26SegPostprocessor>(
            input_width, input_height,
            score_threshold_, nms_threshold_,
            is_ort_configured
        );
    }

private:
    float score_threshold_;
    float nms_threshold_;
};

}  // namespace dxapp

#endif  // YOLO26S_SEG_FACTORY_HPP

### 5.4. Asynchronous result workers

Each worker creates its own `InferenceEngine`, reads the model input shape, creates task-specific processors, and registers an asynchronous callback. The callback post-processes and renders the latest result before posting it to the Qt window.

In [10]:
show_source("yolo26s_3.cpp", "void ResultWorker<ResultT, FactoryT>::run()", line_count=92)

yolo26s_3.cpp:1236-1327


void ResultWorker<ResultT, FactoryT>::run() {
    try {
        dxrt::InferenceOption io;
        io.bufferCount = max_inflight_;
        dxrt::InferenceEngine ie(model_path_, io);
        if (!dxapp::minversionforRTandCompiler(&ie)) {
            throw std::runtime_error(name_ + " model/runtime version mismatch: " + model_path_);
        }

        auto input_shape = ie.GetInputs().front().shape();
        int input_width = 0;
        int input_height = 0;
        parseInputShape(input_shape, input_width, input_height);
        bool is_float_input = (ie.GetInputs().front().type() == dxrt::DataType::FLOAT);
        bool is_nhwc = isInputNHWC(input_shape);

        FactoryT factory;
        auto preprocessor = factory.createPreprocessor(input_width, input_height);
        std::shared_ptr<dxapp::IPostprocessor<ResultT>> postprocessor(
            factory.createPostprocessor(input_width, input_height, ie.IsOrtConfigured()));

        std::cout << "[INFO] " << name_ << " model: " << model_path_ << std::endl;
        std::cout << "[INFO] " << name_ << " input size (WxH): "
                  << input_width << "x" << input_height << std::endl;
        std::cout << "[INFO] " << name_ << " async queue size: "
                  << max_inflight_ << std::endl;

        auto timing = std::make_shared<TimingReporter>(name_, timing_enabled_, timing_interval_ms_);
        int last_job_id = -1;

        ie.RegisterCallback([this, postprocessor, timing](
                            dxrt::TensorPtrs& outputs, void* user_data) -> int {
            auto job = std::unique_ptr<AsyncJob>(static_cast<AsyncJob*>(user_data));
            if (!job) {
                releaseInflightSlot();
                return -1;
            }

            auto callback_time = std::chrono::steady_clock::now();
            try {
                auto post_begin = std::chrono::steady_clock::now();
                std::vector<ResultT> results;
                cv::Mat rendered;
                std::chrono::steady_clock::time_point post_end;
                std::chrono::steady_clock::time_point render_end;

                {
                    std::lock_guard<std::mutex> lock(callback_mutex_);
                    results = postprocessor->process(outputs, job->ctx);
                    post_end = std::chrono::steady_clock::now();
                    if constexpr (std::is_same_v<ResultT, dxapp::InstanceSegmentationResult>) {
                        rendered = renderSegmentationFast(job->frame, results, job->ctx, seg_render_options_);
                    } else if constexpr (std::is_same_v<ResultT, dxapp::DetectionResult>) {
                        rendered = renderDetectionsThinText(job->frame, results);
                    } else if constexpr (std::is_same_v<ResultT, dxapp::PoseResult>) {
                        rendered = renderPoseThinText(job->frame, results, job->ctx);
                    }
                    render_end = std::chrono::steady_clock::now();
                    timing->add(msBetween(job->preprocess_begin, job->preprocess_end),
                                msBetween(job->submit_time, callback_time),
                                msBetween(post_begin, post_end),
                                msBetween(post_end, render_end),
                                msBetween(job->preprocess_begin, render_end));
                }

                if (running_.load(std::memory_order_relaxed) && !rendered.empty()) {
                    window_->postFrame(panel_index_, rendered);
                }
            } catch (const std::exception& e) {
                running_.store(false, std::memory_order_relaxed);
                queue_.wake();
                window_->postError(name_ + " async callback failed: " + e.what());
            }

            releaseInflightSlot();
            return 0;
        });

        while (running_.load(std::memory_order_relaxed)) {
            cv::Mat frame;
            if (!queue_.waitPop(frame, running_)) {
                continue;

### 5.5. Camera and video capture

`CaptureThread` uses OpenCV `VideoCapture`. Video input loops unless `--no-loop-video` is set. Camera input uses the V4L2 backend and requests MJPG format.

In [11]:
show_source("yolo26s_3.cpp", "void CaptureThread::runVideo()", line_count=65)
show_source("yolo26s_3.cpp", "void CaptureThread::runCamera()", line_count=65)

yolo26s_3.cpp:1374-1438


void CaptureThread::runVideo() {
    cv::VideoCapture cap(args_.video);
    if (!cap.isOpened()) {
        window_->postError("Could not open video file: " + args_.video);
        return;
    }

    double source_fps = cap.get(cv::CAP_PROP_FPS);
    if (source_fps <= 1e-3) {
        source_fps = 30.0;
    }
    const auto frame_delay = std::chrono::duration<double>(1.0 / source_fps);

    while (running_.load(std::memory_order_relaxed)) {
        auto loop_start = std::chrono::steady_clock::now();
        cv::Mat frame;
        bool ok = cap.read(frame);
        if (!ok || frame.empty()) {
            if (args_.no_loop_video) {
                window_->postError("Video ended.");
                break;
            }

            cap.set(cv::CAP_PROP_POS_FRAMES, 0);
            ok = cap.read(frame);
            if (!ok || frame.empty()) {
                cap.release();
                cap.open(args_.video);
                if (!cap.isOpened()) {
                    window_->postError("Video loop failed to reopen file.");
                    break;
                }
                ok = cap.read(frame);
            }
            if (!ok || frame.empty()) {
                window_->postError("Video file has no readable frames.");
                break;
            }
        }

        publishFrame(frame);

        auto elapsed = std::chrono::steady_clock::now() - loop_start;
        if (elapsed < frame_delay && running_.load(std::memory_order_relaxed)) {
            std::this_thread::sleep_for(frame_delay - elapsed);
        }
    }
}

void CaptureThread::runCamera() {
    cv::VideoCapture cap;
    bool opened = false;

    if (!args_.device.empty() && args_.device[0] == '/') {
        opened = cap.open(args_.device, cv::CAP_V4L2);
    }
    if (!opened) {
        int index = parseCameraIndex(args_.device);
        opened = cap.open(index, cv::CAP_V4L2);
    }
    if (!opened) {
        window_->postError("VideoCapture failed for camera: " + args_.device);
        return;
    }

yolo26s_3.cpp:1423-1487


void CaptureThread::runCamera() {
    cv::VideoCapture cap;
    bool opened = false;

    if (!args_.device.empty() && args_.device[0] == '/') {
        opened = cap.open(args_.device, cv::CAP_V4L2);
    }
    if (!opened) {
        int index = parseCameraIndex(args_.device);
        opened = cap.open(index, cv::CAP_V4L2);
    }
    if (!opened) {
        window_->postError("VideoCapture failed for camera: " + args_.device);
        return;
    }

    cap.set(cv::CAP_PROP_FOURCC, cv::VideoWriter::fourcc('M', 'J', 'P', 'G'));

    if (args_.width > 0) {
        cap.set(cv::CAP_PROP_FRAME_WIDTH, args_.width);
    }
    if (args_.height > 0) {
        cap.set(cv::CAP_PROP_FRAME_HEIGHT, args_.height);
    }
    if (args_.fps > 0) {
        cap.set(cv::CAP_PROP_FPS, args_.fps);
    }

    std::cout << "[INFO] Input: USB webcam " << args_.device << std::endl;
    std::cout << "[INFO] Camera resolution: "
              << cap.get(cv::CAP_PROP_FRAME_WIDTH) << "x"
              << cap.get(cv::CAP_PROP_FRAME_HEIGHT) << std::endl;

    while (running_.load(std::memory_order_relaxed)) {
        cv::Mat frame;
        if (!cap.read(frame) || frame.empty()) {
            window_->postError("cap.read() failed or stream ended.");
            break;
        }
        publishFrame(frame);
    }
}

void CaptureThread::run() {
    try {
        if (!args_.video.empty()) {
            std::cout << "[INFO] Input: video file " << args_.video << std::endl;
            if (args_.no_loop_video) {
                std::cout << "[INFO] Video will not loop (stop at EOF)" << std::endl;
            }
            runVideo();
        } else {
            runCamera();
        }
    } catch (const std::exception& e) {
        window_->postError(std::string("Capture thread failed: ") + e.what());
    }
}

}  // namespace

int main(int argc, char* argv[]) {
    try {
        AppArgs args = parseArgs(argc, argv);
        args.model = absolutePath(args.model);

### 5.6. Qt 2 x 2 window

`QuadWindow` creates the four panels, starts the three result workers and capture thread, updates FPS labels, and performs an orderly shutdown.

In [12]:
show_source("yolo26s_3.cpp", "class QuadWindow final", line_count=88)

yolo26s_3.cpp:936-1023


class QuadWindow final : public QMainWindow {
public:
    explicit QuadWindow(const AppArgs& args)
        : args_(args), capture_(args, this) {
        setWindowTitle("yolo26s_3");

        auto* central = new QWidget;
        central->setFocusPolicy(Qt::StrongFocus);
        setCentralWidget(central);

        auto* grid = new QGridLayout(central);
        grid->setSpacing(0);
        grid->setContentsMargins(0, 0, 0, 0);
        grid->setColumnStretch(0, 1);
        grid->setColumnStretch(1, 1);
        grid->setRowStretch(0, 1);
        grid->setRowStretch(1, 1);

        const int cells[kPanelCount][2] = {{0, 0}, {0, 1}, {1, 0}, {1, 1}};
        for (int i = 0; i < kPanelCount; ++i) {
            QLabel* image = nullptr;
            QLabel* fps = nullptr;
            QWidget* panel = (i == kDemoPanel)
                ? makeDemoImagePanel(&image, &fps)
                : makePanel(kPanelTitles[i],
                            &image,
                            &fps,
                            args_.show_exit_button && i == kPosePanel);
            grid->addWidget(panel, cells[i][0], cells[i][1]);
            image_labels_.push_back(image);
            fps_labels_.push_back(fps);
            fps_counts_.push_back(0);
        }

        cv::Mat demo_image = cv::imread(args_.demo_image, cv::IMREAD_COLOR);
        if (demo_image.empty()) {
            throw std::runtime_error("[ERROR] Could not read demo image: " + args_.demo_image);
        }
        setPanelFrame(kDemoPanel, demo_image, false);

        fps_timer_ = new QTimer(this);
        fps_timer_->setInterval(1000);
        QObject::connect(fps_timer_, &QTimer::timeout, this, [this] { tickFps(); });
        fps_timer_->start();

        workers_.push_back(std::make_unique<ResultWorker<dxapp::DetectionResult, dxapp::Yolo26sFactory>>(
            kOdPanel, "Object Detection", args_.model, kOdAsyncQueueSize, this));
        workers_.push_back(std::make_unique<ResultWorker<dxapp::PoseResult, dxapp::Yolo26s_poseFactory>>(
            kPosePanel, "Pose Estimation", args_.model_pose, kPoseAsyncQueueSize, this));
        SegRenderOptions seg_render_options;
        seg_render_options.width = args_.seg_render_width;
        seg_render_options.height = args_.seg_render_height;
        seg_render_options.bbox_only = args_.debug_seg_bbox_only;
        workers_.push_back(std::make_unique<ResultWorker<dxapp::InstanceSegmentationResult, dxapp::Yolo26s_segFactory>>(
            kSegPanel,
            "Instance Segmentation",
            args_.model_seg,
            kSegAsyncQueueSize,
            this,
            args_.debug_seg_timing,
            args_.debug_timing_interval_ms,
            seg_render_options));

        std::vector<IFrameConsumer*> consumers;
        consumers.reserve(workers_.size());
        for (auto& worker : workers_) {
            worker->start();
            consumers.push_back(worker.get());
        }
        capture_.setConsumers(std::move(consumers));
        capture_.start();
        
        // Initialize video writers if saving is enabled
        if (args_.save_video && !args_.output_video.empty()) {
            initializeVideoWriters();
        }
    }

    ~QuadWindow() override {
        shutdown();
    }

    void postFrame(int panel_index, const cv::Mat& frame) {
        if (closing_.load(std::memory_order_relaxed)) {
            return;
        }
        cv::Mat safe_frame = frame;
        QMetaObject::invokeMethod(this, [this, panel_index, safe_frame]() mutable {

## 6. Build the application

`build.sh` creates `app/build/`, configures CMake in Release mode, and runs `make` with all CPU cores reported by `nproc`.

In [13]:
build_result = subprocess.run(
    ["./build.sh"],
    cwd=APP_ROOT,
    check=False,
)

binary_path = APP_ROOT / "build" / "yolo26s_3"
print(f"Build exit code: {build_result.returncode}")
print(f"Executable exists: {binary_path.is_file()}")

-- Configuring done (0.0s)
-- Generating done (0.0s)
-- Build files have been written to: /home/donggyun/git/dx-tutorials/notebooks/T21-demo-yolo26-od-pos-seg/app/build
[ 50%] Building CXX object CMakeFiles/yolo26s_3.dir/yolo26s_3.cpp.o
[100%] Linking CXX executable yolo26s_3
[100%] Built target yolo26s_3
Build exit code: 0
Executable exists: True


## 7. Run the camera demo

The default device is `/dev/video0` at a requested 1280 x 720 and 30 FPS. Change the variables below to select another camera or capture setting. The run cell opens a full-screen Qt window and blocks until the application exits.

In [14]:
CAMERA_DEVICE = "/dev/video0"
CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720
CAMERA_FPS = 30

camera_path = Path(CAMERA_DEVICE)
print(f"Camera exists: {camera_path.exists()}")
if shutil.which("v4l2-ctl"):
    subprocess.run(["v4l2-ctl", "--list-devices"], check=False)

Camera exists: True
Integrated Camera: Integrated C (usb-0000:00:14.0-11):
	/dev/video0
	/dev/video1
	/dev/media0



In [15]:
if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")
if not camera_path.exists():
    raise FileNotFoundError(f"The camera is not available: {CAMERA_DEVICE}")

subprocess.run(
    [
        "./run_camera.sh",
        "--camera",
        CAMERA_DEVICE,
        "--width",
        str(CAMERA_WIDTH),
        "--height",
        str(CAMERA_HEIGHT),
        "--fps",
        str(CAMERA_FPS),
    ],
    cwd=APP_ROOT,
    check=False,
)

[INFO] Input: USB webcam /dev/video0
[INFO] Camera resolution: 1280x720
[INFO] Object Detection model: /home/donggyun/git/dx-tutorials/notebooks/T21-demo-yolo26-od-pos-seg/assets/models/yolo26s.dxnn
[INFO] Object Detection input size (WxH): 640x640
[INFO] Object Detection async queue size: 2
[INFO] Instance Segmentation model: /home/donggyun/git/dx-tutorials/notebooks/T21-demo-yolo26-od-pos-seg/assets/models/yolo26s-seg.dxnn
[INFO] Instance Segmentation input size (WxH): 640x640
[INFO] Instance Segmentation async queue size: 3
[INFO] Pose Estimation model: /home/donggyun/git/dx-tutorials/notebooks/T21-demo-yolo26-od-pos-seg/assets/models/yolo26s-pose.dxnn
[INFO] Pose Estimation input size (WxH): 640x640
[INFO] Pose Estimation async queue size: 2


CompletedProcess(args=['./run_camera.sh', '--camera', '/dev/video0', '--width', '1280', '--height', '720', '--fps', '30'], returncode=0)

## 8. Run the video demo

Set `VIDEO_PATH` to a file under `assets/videos/`. `run_video.sh` accepts the video path as its first argument.

In [ ]:
VIDEO_PATH = video_files[0] if video_files else None
print(f"Selected video: {VIDEO_PATH if VIDEO_PATH else 'no video available'}")

In [ ]:
if not binary_path.is_file():
    raise FileNotFoundError("Build the application before running the demo.")
if VIDEO_PATH is None or not VIDEO_PATH.is_file():
    raise FileNotFoundError("Set VIDEO_PATH to an existing video file.")

subprocess.run(
    ["./run_video.sh", str(VIDEO_PATH)],
    cwd=APP_ROOT,
    check=False,
)

## Controls

- `Esc` or `q`: exit
- `EXIT` button: exit with the mouse

## Troubleshooting

- **Qt5 is not found:** install `qtbase5-dev` and run `./build.sh --clean`.
- **DXRT is not found:** verify the DXRT SDK installation and run `dxrt-cli -s`.
- **A model or image is missing:** place the models under `assets/models/` and the demo image at `assets/yolo26.png`, or pass explicit paths.
- **The camera cannot be opened:** verify the V4L2 device and user permissions.
- **The window does not appear:** use a graphical desktop, remote desktop, or correctly configured X11 forwarding.
- **The notebook cell remains busy:** the run cell blocks while the Qt event loop is active; close the window to finish the cell.

## Summary

One capture thread sends each frame to three independent asynchronous DXRT pipelines. Task-specific factories create the preprocessing, post-processing, and visualization components for detection, pose estimation, and instance segmentation. Qt combines their results with a static demo image in a full-screen 2 x 2 layout.